# 04 - LGD, EAD (CCF) & Expected Loss
`EL = PD x LGD x EAD`.

**What is real vs synthetic here:**
- **PD** - real, from the scorecard.
- **EAD / CCF** - estimated on *real* data using a proxy: for defaulters, the share of undrawn headroom (vs. the Jun balance, BILL_AMT4) that was drawn by Sep (BILL_AMT1).
- **LGD** - the dataset has **no recovery data**. The two-stage LGD model is demonstrated on **clearly synthetic recoveries** generated from a known process, so we can verify the pipeline recovers the true parameters. Treat LGD values as illustrative, not as estimates for this portfolio.
- Horizon: the dataset's default flag is **1-month-ahead**; EL is not annualised.

In [1]:
import os, sys; os.chdir('..') if os.path.basename(os.getcwd())=='notebooks' else None; sys.path.insert(0,'.')
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, joblib, matplotlib.pyplot as plt
%matplotlib inline
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
train, test = joblib.load('data/processed/split.joblib'); art = joblib.load('data/processed/scorecard.joblib'); sc = art['scorecard']
sig = lambda z: 1/(1+np.exp(-z))
def ref_util(d): return (d.BILL_AMT4.clip(lower=0) / d.LIMIT_BAL).clip(0, 1.5)
d = train[train.default == 1]; ref = d.BILL_AMT4.clip(lower=0); room = d.LIMIT_BAL - ref; ok = room > 0.10 * d.LIMIT_BAL
ccf = ((d.BILL_AMT1.clip(lower=0) - ref) / room)[ok].clip(0.001, 0.999)
print('defaulters used: %d | mean CCF proxy %.3f | median %.3f' % (len(ccf), ccf.mean(), ccf.median()))
glm = sm.GLM(ccf, sm.add_constant(ref_util(d[ok])), family=sm.families.Binomial()).fit()
print('CCF fractional-logit: util coef %.3f (p=%.3g), pseudo-R2 %.3f' % (glm.params.iloc[1], glm.pvalues.iloc[1], 1 - glm.deviance/glm.null_deviance))
def ead(d):
    c = sig(glm.params.iloc[0] + glm.params.iloc[1]*ref_util(d)); bal = d.BILL_AMT1.clip(lower=0); return bal + c*(d.LIMIT_BAL - bal).clip(lower=0)

defaulters used: 3792 | mean CCF proxy 0.178 | median 0.001
CCF fractional-logit: util coef 1.724 (p=7.74e-36), pseudo-R2 0.061


## Two-stage LGD on synthetic recoveries (known data-generating process)

In [2]:
rng = np.random.default_rng(0)
def feats(d): return np.c_[d.util.clip(0, 1.5), d.LIMIT_BAL / 1e6]
def true_p(d): f = feats(d); return sig(0.8 - 2.0*f[:,0] + 3.0*f[:,1])
def true_m(d):  f = feats(d); return sig(-0.2 - 0.8*f[:,0] + 2.0*f[:,1])
def simulate(d):
    rec = rng.random(len(d)) < true_p(d); m = true_m(d); k = 8
    rate = rng.beta(m*k, (1-m)*k); return np.where(rec, rate, 0.0)
dtr = train[train.default == 1]; r = simulate(dtr); pos = r > 0
st1 = LogisticRegression().fit(feats(dtr), pos)
X2 = sm.add_constant(feats(dtr)[pos]); st2 = sm.GLM(np.clip(r[pos], 1e-3, 1-1e-3), X2, family=sm.families.Binomial()).fit()
lgd_hat = lambda d: 1 - st1.predict_proba(feats(d))[:,1] * sig(sm.add_constant(feats(d), has_constant='add') @ st2.params)
lgd_true = lambda d: 1 - true_p(d) * true_m(d)
print('Stage-1 AUC (any recovery): %.3f' % roc_auc_score(pos, st1.predict_proba(feats(dtr))[:,1]))
print('Stage-2 params est %s vs true [-0.2,-0.8,2.0]' % np.round(st2.params, 2))
print('LGD recovered: corr %.3f, RMSE %.3f (true mean %.3f, est mean %.3f)' % (np.corrcoef(lgd_hat(dtr), lgd_true(dtr))[0,1], np.sqrt(np.mean((lgd_hat(dtr)-lgd_true(dtr))**2)), lgd_true(dtr).mean(), lgd_hat(dtr).mean()))

Stage-1 AUC (any recovery): 0.722
Stage-2 params est [-0.18 -0.83  1.91] vs true [-0.2,-0.8,2.0]
LGD recovered: corr 0.999, RMSE 0.009 (true mean 0.742, est mean 0.746)


## Expected Loss on the test portfolio

In [3]:
s = sc.score(test); t = test.assign(score=s, PD=sc.pd_from_score(s), LGD=lgd_hat(test), EAD=ead(test)); t['EL'] = t.PD * t.LGD * t.EAD
real = (t.default * lgd_true(test) * t.EAD).sum()
print('Portfolio EAD: NT$ {:,.0f}'.format(t.EAD.sum())); print('Predicted EL: NT$ {:,.0f}  ({:.2%} of EAD)'.format(t.EL.sum(), t.EL.sum()/t.EAD.sum())); print('Realised loss (actual defaults x synthetic true LGD): NT$ {:,.0f}  -> predicted/realised = {:.2f}'.format(real, t.EL.sum()/real))
t['band'] = pd.qcut(t.score, 5, labels=['Q1 worst','Q2','Q3','Q4','Q5 best']); display(t.groupby('band').agg(accounts=('EL','size'), mean_PD=('PD','mean'), mean_LGD=('LGD','mean'), EAD=('EAD','sum'), EL=('EL','sum')).round(3))
joblib.dump(dict(glm=glm), 'data/processed/ead_model.joblib')

Portfolio EAD: NT$ 596,171,133
Predicted EL: NT$ 85,381,114  (14.32% of EAD)
Realised loss (actual defaults x synthetic true LGD): NT$ 91,517,779  -> predicted/realised = 0.93


,accounts,mean_PD,mean_LGD,EAD,EL
band,,,,,
Q1 worst,1800,0.562,0.775,9.804217e+07,4.414721e+07
Q2,1800,0.216,0.736,7.072953e+07,1.168671e+07
Q3,1801,0.139,0.734,8.659202e+07,9.316710e+06
Q4,1799,0.108,0.661,1.241784e+08,9.434602e+06
Q5 best,1800,0.075,0.609,2.166290e+08,1.079588e+07


['data/processed/ead_model.joblib']